In [1]:
import pandas as pd
import numpy as np

In [ ]:
fusion_df = pd.read_csv("/scratch/yag1/ego4d_data/v2/baseline_qwen_results.csv")

In [ ]:
import ast

fusion_df["video_retrieval_timestamps"] = fusion_df["video_retrieval_timestamps"].apply(ast.literal_eval)
fusion_df["audio_retrieval_timestamps"] = fusion_df["audio_retrieval_timestamps"].apply(ast.literal_eval)
fusion_df["clap_distances"] = fusion_df["clap_distances"].apply(ast.literal_eval)
fusion_df["qwen_distances"] = fusion_df["qwen_distances"].apply(ast.literal_eval)

In [4]:
from collections import defaultdict

In [5]:
def generate_rank_mapping(video_sections, audio_sections):
    rank_mapping = defaultdict(list)
    
    for audio_rank, audio_sec in enumerate(audio_sections, start=1):
        # if len(rank_mapping[tuple(audio_sec)]) == 0:
        rank_mapping[tuple(audio_sec)].append(audio_rank)
        # else:
        #     assert(audio_rank > rank_mapping[tuple(audio_sec)][0]), f"The rank of the new proposed audio section {audio_sec} with rank {audio_rank} is better than the existing rank {rank_mapping[tuple(audio_sec)][0]} for the same section."
            # rank_mapping[tuple(audio_sec)][0] = min(rank_mapping[tuple(audio_sec)][0], audio_rank)

    for video_rank, video_sec in enumerate(video_sections, start=1):
        # if len(rank_mapping[tuple(video_sec)]) == 1:
        rank_mapping[tuple(video_sec)].append(video_rank)
        # else:
        #     assert(video_rank > rank_mapping[tuple(video_sec)][1]), f"The rank of the new proposed video section {video_sec} with rank {video_rank} is better than the existing rank {rank_mapping[tuple(video_sec)][1]} for the same section."
            # rank_mapping[tuple(video_sec)][1] = min(rank_mapping[tuple(video_sec)][1], video_rank)

    return rank_mapping

In [6]:
from scipy.stats import spearmanr

In [7]:

def generate_correlation(video_sections, audio_sections):
    rank_mapping = generate_rank_mapping(video_sections, audio_sections)
    
    audio_ranks = []
    video_ranks = []
        
    for sec in rank_mapping:
        assert len(rank_mapping[sec]) == 2, f"Section {sec} does not have both audio and video ranks"
        audio_ranks.append(rank_mapping[sec][0])
        video_ranks.append(rank_mapping[sec][1])    
    
    correlation = spearmanr(audio_ranks, video_ranks).correlation
    
    return correlation

In [8]:
fusion_df["correlation"] = fusion_df.apply(lambda row: generate_correlation(row["video_retrieval_timestamps"], row["audio_retrieval_timestamps"]), axis=1)

In [9]:
fusion_df["correlation"].describe()

count    55.000000
mean      0.051458
std       0.141428
min      -0.376825
25%      -0.052964
50%       0.058832
75%       0.153271
max       0.309472
Name: correlation, dtype: float64

In [10]:
def generate_rrf_rank(video_sections, audio_sections, weight_alpha):
    rank_mapping = generate_rank_mapping(video_sections, audio_sections)
    
    k = 60
    
    rrf_ranking = [] 

    for section, ranks in rank_mapping.items():
        assert(len(ranks) == 2)
        rrf_score = weight_alpha / (k + ranks[0]) + (1-weight_alpha) / (k + ranks[1])
        rrf_ranking.append((rrf_score, section))
    
    rrf_ranking.sort(key=lambda x: x[0], reverse=True)
    
    # Extract the ranked sections
    ranked_sections = [list(item[1]) for item in rrf_ranking]
        
    scores = [item[0] for item in rrf_ranking]
    
    return ranked_sections, scores 

In [11]:
def compute_IoU(pred, gt_start, gt_end):
    """Compute the IoU given predicted and ground truth windows."""
    gt = [[gt_start, gt_end]]
    assert isinstance(pred, list) and isinstance(gt, list)
    pred_is_list = isinstance(pred[0], list)
    gt_is_list = isinstance(gt[0], list)
    if not pred_is_list:
        pred = [pred]
    if not gt_is_list:
        gt = [gt]
    pred, gt = np.array(pred), np.array(gt)
    inter_left = np.maximum(pred[:, 0, None], gt[None, :, 0])
    inter_right = np.minimum(pred[:, 1, None], gt[None, :, 1])
    inter = np.maximum(0.0, inter_right - inter_left)
    union_left = np.minimum(pred[:, 0, None], gt[None, :, 0])
    union_right = np.maximum(pred[:, 1, None], gt[None, :, 1])
    union = np.maximum(0.0, union_right - union_left)
    overlap = 1.0 * inter / union
    if not gt_is_list:
        overlap = overlap[:, 0]
    if not pred_is_list:
        overlap = overlap[0]
    return overlap


In [12]:
def compute_recall_at_k(k_val, IoU_threshold, iou_list):
    """Compute recall at k given a list of IoU values."""
    
    top_k_iou = iou_list[:k_val]
    num_relevant = sum(iou >= IoU_threshold for iou in top_k_iou)
    return int((num_relevant > 0).item())

In [13]:
def temporal_iou(seg1, seg2):
    s1, e1 = seg1
    s2, e2 = seg2

    inter = max(0, min(e1, e2) - max(s1, s2))
    union = (e1 - s1) + (e2 - s2) - inter

    return inter / union if union > 0 else 0

In [14]:
def temporal_iom(seg1, seg2):
    s1, e1 = seg1
    s2, e2 = seg2
    
    inter = max(0, min(e1, e2) - max(s1, s2))
    
    len1 = e1 - s1
    len2 = e2 - s2
    
    return inter / min(len1, len2) if min(len1, len2) > 0 else 0

In [15]:
def temporal_NMS(timestamps, use_iou=True, threshold=0.5):
    """Apply temporal NMS to a list of timestamps."""
    keep = []
    
    segments = timestamps.copy()

    while segments:
        best = segments[0]
        keep.append(best)

        remaining = []

        for seg in segments[1:]:
            if use_iou:
                metric = temporal_iou(
                    best,
                    seg
                )
            else:
                metric = temporal_iom(
                    best,
                    seg
                )

            if metric < threshold:
                remaining.append(seg)

        segments = remaining

    return keep
    

In [16]:
import math

In [17]:
def soft_nms(segments, scores, iom_threshold=0.5, use_linear=True, use_iom=True, sigma=0.5, score_threshold=0.001):
    segments = list(zip(segments, scores))
    keep = []

    while segments:

        # Sort by current score
        segments.sort(key=lambda x: x[1], reverse=True)

        best_seg, best_score = segments[0]
        keep.append((best_seg, best_score))

        updated = []

        for seg, score in segments[1:]:

            overlap = temporal_iou(best_seg, seg) if not use_iom else temporal_iom(best_seg, seg)

            if use_linear:
                if overlap > iom_threshold:
                    score *= (1 - overlap)
            else:
                score *= math.exp(- (overlap ** 2) / sigma)
            
            if score > score_threshold:
                updated.append((seg, score))

        segments = updated

    return [seg for seg, _ in keep]

In [18]:
alpha_values = [i for i in np.arange(0.05, 1.0, 0.05)]
# alpha_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

In [19]:
print("Starting RRF calculations")

Starting RRF calculations


In [20]:
from tqdm import tqdm

In [ ]:
for alpha in tqdm(alpha_values):
    # Generate the ranked sections using the RRF method
    fusion_df[[f"fusion_rrf_ranked_{alpha}_timestamps",f"fusion_rrf_ranked_{alpha}_scores"]] = fusion_df.apply(lambda row: pd.Series(generate_rrf_rank(row["video_retrieval_timestamps"],row["audio_retrieval_timestamps"], alpha)), axis=1)   
     
     #NMS based filtering (IoU)
    fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_timestamps"] = fusion_df.apply(lambda row: temporal_NMS(row[f"fusion_rrf_ranked_{alpha}_timestamps"]), axis=1)
    
    #NMS based filtering (IoM)
    fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoM_timestamps"] = fusion_df.apply(lambda row: temporal_NMS(row[f"fusion_rrf_ranked_{alpha}_timestamps"], use_iou=False), axis=1)
    
    # soft NMS based filtering (linear IoM)
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_rrf_ranked_{alpha}_timestamps"], row[f"fusion_rrf_ranked_{alpha}_scores"]), axis=1)
    
    # soft NMS based filtering (gaussian IoM)
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_rrf_ranked_{alpha}_timestamps"], row[f"fusion_rrf_ranked_{alpha}_scores"], use_linear=False), axis=1)
    
    # soft NMS based filtering (linear IoU)
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_rrf_ranked_{alpha}_timestamps"], row[f"fusion_rrf_ranked_{alpha}_scores"], use_iom=False), axis=1)

    # soft NMS based filtering (gaussian IoU)
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_rrf_ranked_{alpha}_timestamps"], row[f"fusion_rrf_ranked_{alpha}_scores"], use_iom=False, use_linear=False), axis=1)
    
    # calculate the IoU for the ranked sections and store in a new column
    fusion_df[f"fusion_rrf_ranked_{alpha}_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_NMS_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoM_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_NMS_IoM_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
        
    # calculate recall at different IoU thresholds & sizes and store in new columns
    for k_size in [3, 5]:
        for threshold in [0.3, 0.5]:
            fusion_df[f"fusion_rrf_ranked_{alpha}_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_IoU"]), axis=1)
            
            fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_NMS_IoU"]), axis=1)
            
            fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoM_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_NMS_IoM_IoU"]), axis=1)
            
            fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_IoU"]), axis=1)
            
            fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_IoU"]), axis=1)
            
            fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_IoU"]), axis=1)
            
            fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_IoU"]), axis=1)
                    
        # average recall at k across different IoU thresholds
        fusion_df[f"fusion_rrf_ranked_{alpha}_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoM_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoM_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_NMS_IoM_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoM_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoM_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_linear_IoU_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_rrf_ranked_{alpha}_soft_NMS_gaussian_IoU_recall_at_{k_size}_0.5"]) / 2
        

In [22]:
print("Finished RRF calculations")

Finished RRF calculations


In [23]:
print("Baseline results:\n\n")

Baseline results:




In [24]:
cols = [
            "video_IoU_recall_at_3_0.3", "audio_IoU_recall_at_3_0.3",
              "video_IoU_recall_at_3_0.5", "audio_IoU_recall_at_3_0.5",
              "video_IoU_recall_at_5_0.3", "audio_IoU_recall_at_5_0.3",
              "video_IoU_recall_at_5_0.5", "audio_IoU_recall_at_5_0.5",
              
             "video_temporal_NMS_recall_at_3_0.3", "audio_temporal_NMS_recall_at_3_0.3", 
              "video_temporal_NMS_recall_at_3_0.5", "audio_temporal_NMS_recall_at_3_0.5",
              "video_temporal_NMS_recall_at_5_0.3", "audio_temporal_NMS_recall_at_5_0.3",
              "video_temporal_NMS_recall_at_5_0.5", "audio_temporal_NMS_recall_at_5_0.5",
              
              "video_temporal_NMS_IoM_recall_at_3_0.3", "audio_temporal_NMS_IoM_recall_at_3_0.3",
              "video_temporal_NMS_IoM_recall_at_3_0.5", "audio_temporal_NMS_IoM_recall_at_3_0.5",
              "video_temporal_NMS_IoM_recall_at_5_0.3", "audio_temporal_NMS_IoM_recall_at_5_0.3",
              "video_temporal_NMS_IoM_recall_at_5_0.5", "audio_temporal_NMS_IoM_recall_at_5_0.5",
              
              "video_temporal_soft_NMS_linear_IoM_recall_at_3_0.3", "audio_temporal_soft_NMS_linear_IoM_recall_at_3_0.3",
                "video_temporal_soft_NMS_linear_IoM_recall_at_3_0.5", "audio_temporal_soft_NMS_linear_IoM_recall_at_3_0.5",
                "video_temporal_soft_NMS_linear_IoM_recall_at_5_0.3", "audio_temporal_soft_NMS_linear_IoM_recall_at_5_0.3",
                "video_temporal_soft_NMS_linear_IoM_recall_at_5_0.5", "audio_temporal_soft_NMS_linear_IoM_recall_at_5_0.5",
                
                "video_temporal_soft_NMS_gaussian_IoM_recall_at_3_0.3", "audio_temporal_soft_NMS_gaussian_IoM_recall_at_3_0.3",
                "video_temporal_soft_NMS_gaussian_IoM_recall_at_3_0.5", "audio_temporal_soft_NMS_gaussian_IoM_recall_at_3_0.5",
                "video_temporal_soft_NMS_gaussian_IoM_recall_at_5_0.3", "audio_temporal_soft_NMS_gaussian_IoM_recall_at_5_0.3",
                "video_temporal_soft_NMS_gaussian_IoM_recall_at_5_0.5", "audio_temporal_soft_NMS_gaussian_IoM_recall_at_5_0.5",
                
                "video_temporal_soft_NMS_linear_IoU_recall_at_3_0.3", "audio_temporal_soft_NMS_linear_IoU_recall_at_3_0.3",
                "video_temporal_soft_NMS_linear_IoU_recall_at_3_0.5", "audio_temporal_soft_NMS_linear_IoU_recall_at_3_0.5",
                "video_temporal_soft_NMS_linear_IoU_recall_at_5_0.3", "audio_temporal_soft_NMS_linear_IoU_recall_at_5_0.3",
                "video_temporal_soft_NMS_linear_IoU_recall_at_5_0.5", "audio_temporal_soft_NMS_linear_IoU_recall_at_5_0.5",
                
                "video_temporal_soft_NMS_gaussian_IoU_recall_at_3_0.3", "audio_temporal_soft_NMS_gaussian_IoU_recall_at_3_0.3",
                "video_temporal_soft_NMS_gaussian_IoU_recall_at_3_0.5", "audio_temporal_soft_NMS_gaussian_IoU_recall_at_3_0.5",
                "video_temporal_soft_NMS_gaussian_IoU_recall_at_5_0.3", "audio_temporal_soft_NMS_gaussian_IoU_recall_at_5_0.3",
                "video_temporal_soft_NMS_gaussian_IoU_recall_at_5_0.5", "audio_temporal_soft_NMS_gaussian_IoU_recall_at_5_0.5"
    
    ]


In [25]:
means = fusion_df[cols].mean().sort_values(ascending=False)

for col, avg in means.items():
    print(f"{col}: {avg:.4f}")

audio_temporal_NMS_IoM_recall_at_5_0.3: 0.1273
audio_temporal_soft_NMS_gaussian_IoM_recall_at_5_0.3: 0.1273
audio_temporal_soft_NMS_gaussian_IoU_recall_at_5_0.3: 0.1273
video_temporal_NMS_recall_at_5_0.5: 0.1091
audio_temporal_NMS_recall_at_3_0.3: 0.1091
audio_temporal_NMS_recall_at_5_0.3: 0.1091
video_temporal_NMS_IoM_recall_at_5_0.3: 0.1091
video_temporal_NMS_recall_at_5_0.3: 0.1091
audio_temporal_soft_NMS_linear_IoM_recall_at_3_0.3: 0.1091
video_temporal_soft_NMS_gaussian_IoM_recall_at_3_0.3: 0.1091
video_temporal_soft_NMS_linear_IoM_recall_at_5_0.3: 0.1091
audio_temporal_soft_NMS_linear_IoU_recall_at_5_0.3: 0.1091
audio_temporal_soft_NMS_gaussian_IoU_recall_at_3_0.3: 0.1091
video_temporal_soft_NMS_gaussian_IoU_recall_at_5_0.3: 0.1091
video_temporal_soft_NMS_gaussian_IoU_recall_at_3_0.3: 0.1091
video_temporal_soft_NMS_linear_IoU_recall_at_5_0.3: 0.1091
video_temporal_soft_NMS_linear_IoM_recall_at_3_0.3: 0.1091
audio_temporal_soft_NMS_linear_IoM_recall_at_5_0.3: 0.1091
video_temporal

In [26]:
print("RRF Fusion results:\n\n")

RRF Fusion results:




In [ ]:
fusion_rrf_cols = [col for col in fusion_df.columns if col.startswith("fusion_rrf") and ("recall_at" in col)]

fusion_rrf_means = fusion_df[fusion_rrf_cols].mean().sort_values(ascending=False)

for col, avg in fusion_rrf_means.items():
    print(f"{col}: {avg:.4f}")

In [28]:
def generate_score_mapping(video_sections, audio_sections, video_scores, audio_scores):
    score_mapping = defaultdict(list)
    
    for audio_score, audio_sec in zip(audio_scores, audio_sections):
        # if len(score_mapping[tuple(audio_sec)]) == 0:
            score_mapping[tuple(audio_sec)].append(audio_score)
            # min_audio_score = min(min_audio_score, audio_score)
            # max_audio_score = max(max_audio_score, audio_score)
        # else:
        #     assert(audio_score <= score_mapping[tuple(audio_sec)][0]), f"The score of the new proposed audio section {audio_sec} with score {audio_score} is better than the existing score {score_mapping[tuple(audio_sec)][0]} for the same section."

    for video_score, video_sec in zip(video_scores, video_sections):
        # if len(score_mapping[tuple(video_sec)]) == 1:
            score_mapping[tuple(video_sec)].append(video_score)
        #     min_video_score = min(min_video_score, video_score)
        #     max_video_score = max(max_video_score, video_score)
        # else:
        #     assert(video_score <= score_mapping[tuple(video_sec)][1]), f"The score of the new proposed video section {video_sec} with score {video_score} is better than the existing score {score_mapping[tuple(video_sec)][1]} for the same section."
            
    return score_mapping
    

In [29]:
def min_max_normalize(video_sections, audio_sections, video_scores, audio_scores, min_audio_score, max_audio_score, min_video_score, max_video_score):
    score_mapping= generate_score_mapping(video_sections, audio_sections, video_scores, audio_scores)
    
    normalized_score_mapping = {}
    
    for sec in score_mapping:
        assert len(score_mapping[sec]) == 2, f"Section {sec} does not have both audio and video scores"
        normalized_video_score = (score_mapping[sec][1] - min_video_score) / (max_video_score - min_video_score) if max_video_score > min_video_score else 0
        normalized_audio_score = (score_mapping[sec][0] - min_audio_score) / (max_audio_score - min_audio_score) if max_audio_score > min_audio_score else 0
        normalized_score_mapping[sec] = [normalized_audio_score, normalized_video_score]
    
    return normalized_score_mapping

In [30]:
def generate_score_rank(video_sections, audio_sections, video_scores, audio_scores, weight_alpha, min_audio_score, max_audio_score, min_video_score, max_video_score):
    normalized_score_mapping = min_max_normalize(video_sections, audio_sections, video_scores, audio_scores, min_audio_score, max_audio_score, min_video_score, max_video_score)
    
    score_ranking = []
    
    for section, scores in normalized_score_mapping.items():
        assert(len(scores) == 2)
        fused_score = weight_alpha * scores[0] + (1 - weight_alpha) * scores[1]
        score_ranking.append((fused_score, section))
    
    score_ranking.sort(key=lambda x: x[0], reverse=True)
    
    ranked_sections = [list(item[1]) for item in score_ranking]
    
    scores = [item[0] for item in score_ranking]
    
    return ranked_sections, scores

In [31]:
print("Starting score fusion calculations")

Starting score fusion calculations


In [ ]:
for alpha in tqdm(alpha_values):
    # Generate the ranked sections using the Score Min-Max Normalization method
    fusion_df[[f"fusion_score_ranked_{alpha}_timestamps",f"fusion_score_ranked_{alpha}_scores"]] = fusion_df.apply(lambda row: pd.Series(generate_score_rank(row["video_retrieval_timestamps"],row["audio_retrieval_timestamps"], row["qwen_distances"], row["clap_distances"], alpha, row["clap_min_distance"], row["clap_max_distance"], row["qwen_min_distance"], row["qwen_max_distance"])), axis=1)   
    
     #NMS based filtering (IoU)
    fusion_df[f"fusion_score_ranked_{alpha}_NMS_timestamps"] = fusion_df.apply(lambda row: temporal_NMS(row[f"fusion_score_ranked_{alpha}_timestamps"]), axis=1)
    
    #NMS based filtering (IoM)
    fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoM_timestamps"] = fusion_df.apply(lambda row: temporal_NMS(row[f"fusion_score_ranked_{alpha}_timestamps"], use_iou=False), axis=1)
    
    # NMS based filtering (soft NMS linear IoM)
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_score_ranked_{alpha}_timestamps"], row[f"fusion_score_ranked_{alpha}_scores"]), axis=1)
    
    # NMS based filtering (soft NMS gaussian IoM)
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_score_ranked_{alpha}_timestamps"], row[f"fusion_score_ranked_{alpha}_scores"], use_linear=False), axis=1)
    
    # NMS based filtering (soft NMS linear IoU)
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_score_ranked_{alpha}_timestamps"], row[f"fusion_score_ranked_{alpha}_scores"], use_iom=False), axis=1)
    
    # NMS based filtering (soft NMS gaussian IoU)
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_timestamps"] = fusion_df.apply(lambda row: soft_nms(row[f"fusion_score_ranked_{alpha}_timestamps"], row[f"fusion_score_ranked_{alpha}_scores"], use_iom=False, use_linear=False), axis=1)
    
    # calculate the IoU for the ranked sections and store in a new column
    fusion_df[f"fusion_score_ranked_{alpha}_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_NMS_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoM_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_NMS_IoM_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
    
    fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_IoU"] = fusion_df.apply(lambda row: compute_IoU(row[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_timestamps"], row["query_start_sec"], row["query_end_sec"]), axis=1)
        
    # calculate recall at different IoU thresholds & sizes and store in new columns
    for k_size in [3, 5]:
        for threshold in [0.3, 0.5]:
            fusion_df[f"fusion_score_ranked_{alpha}_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_IoU"]), axis=1)
            
            fusion_df[f"fusion_score_ranked_{alpha}_NMS_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_NMS_IoU"]), axis=1)
            
            fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoM_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_NMS_IoM_IoU"]), axis=1)
            
            fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_IoU"]), axis=1)
            
            fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_IoU"]), axis=1)
            
            fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_IoU"]), axis=1)
            
            fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_recall_at_{k_size}_{threshold}"] = fusion_df.apply(lambda row: compute_recall_at_k(k_size, threshold, row[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_IoU"]), axis=1)
                    
        # average recall at k across different IoU thresholds
        fusion_df[f"fusion_score_ranked_{alpha}_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_score_ranked_{alpha}_NMS_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_NMS_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_NMS_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoM_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoM_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_NMS_IoM_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoM_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoM_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_linear_IoU_recall_at_{k_size}_0.5"]) / 2
        
        fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_mean_IoU_top_{k_size}"] = (fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_recall_at_{k_size}_0.3"] + fusion_df[f"fusion_score_ranked_{alpha}_soft_NMS_gaussian_IoU_recall_at_{k_size}_0.5"]) / 2

  0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
print("Finished score fusion calculations")

In [ ]:
print("Score Fusion results:\n\n")

In [ ]:
fusion_score_cols = [col for col in fusion_df.columns if col.startswith("fusion_score") and ("recall_at" in col)]

fusion_score_means = fusion_df[fusion_score_cols].mean().sort_values(ascending=False)

for col, avg in fusion_score_means.items():
    print(f"{col}: {avg:.4f}")